In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [2]:
df = pd.read_parquet('../data/processed/df_analysis.parquet')

In [20]:
df.columns

Index(['asset_id', 'invoice_id', 'face_value', 'buyer_tax_id', 'seller_tax_id',
       'maturity_date', 'due_date', 'settled_at', 'created_at',
       'reference_date', 'days_to_payment', 'payment_status', 'face_value_bin',
       'year', 'month', 'year_month', 'tax_id', 'company_status',
       'company_status_date', 'company_creation_date', 'company_size',
       'main_cnae', 'main_cnae_description', 'secondary_cnae_array',
       'legal_nature', 'is_mei', 'city', 'state', 'zipcode', 'tax_id_quod',
       'quod_score', 'presumed_revenue', 'created_at_quod',
       'max_expected_payment_time', 'buyer_is_first_transaction',
       'created_month', 'created_quarter', 'created_day_of_week',
       'created_day_of_month', 'created_year', 'is_month_end',
       'days_until_due', 'is_holiday_period', 'face_value_log',
       'is_large_transaction', 'has_quod_score', 'company_age_days',
       'is_active_company', 'state_avg_days', 'state_median_days',
       'state_std_days', 'state_transac

I will start by transforming 

In [4]:
df['max_expected_payment_time'] = df['due_date'] - df['created_at']
df["is_mei"] = np.where(df['is_mei'] == True, 1, 0)

In [8]:
df['max_expected_payment_time'].describe()

count                        868256
mean     39 days 11:04:10.797921350
std      31 days 05:41:07.836123346
min            -1033 days +00:00:00
25%                 8 days 00:00:00
50%                35 days 00:00:00
75%                56 days 00:00:00
max              1444 days 00:00:00
Name: max_expected_payment_time, dtype: object

In [ ]:
buyer_history = df.groupby('buyer_tax_id').agg({
    'days_to_payment': ['mean', 'std', 'median', 'min', 'max', 'count'],
    'face_value': ['sum', 'mean']
}).reset_index()
buyer_history.columns = [
    'buyer_tax_id', 
    'buyer_avg_days', 'buyer_std_days', 'buyer_median_days', 
    'buyer_min_days', 'buyer_max_days', 'buyer_transaction_count',
    'buyer_total_value', 'buyer_avg_value'
]


In [24]:
# Sort first (critical!)
df = df.sort_values(['buyer_tax_id', 'created_at']).reset_index(drop=True)

# Historical stats using only PAST transactions
df['buyer_avg_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().mean().shift(1)
)
df['buyer_std_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().std().shift(1)
)
df['buyer_median_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().median().shift(1)
)
df['buyer_min_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().min().shift(1)
)
df['buyer_max_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().max().shift(1)
)
df['buyer_transaction_count'] = df.groupby('buyer_tax_id').cumcount()  # count BEFORE this one

# Face value aggregates (safe - not using target variable)
df['buyer_total_value'] = df.groupby('buyer_tax_id')['face_value'].transform(
    lambda x: x.expanding().sum().shift(1)
)
df['buyer_avg_value'] = df.groupby('buyer_tax_id')['face_value'].transform(
    lambda x: x.expanding().mean().shift(1)
)

In [10]:
# On-time rate (% of payments within 30 days)
buyer_ontime = df.groupby('buyer_tax_id').apply(
    lambda x: (x['days_to_payment'] <= 30).mean()
).reset_index(name='buyer_ontime_rate')

# Late payment rate (% > 60 days)
buyer_late = df.groupby('buyer_tax_id').apply(
    lambda x: (x['days_to_payment'] > 60).mean()
).reset_index(name='buyer_late_rate')

# Is first transaction for this buyer?
df['buyer_is_first_transaction'] = df.groupby('buyer_tax_id').cumcount() == 0

/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/1580462645.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  buyer_ontime = df.groupby('buyer_tax_id').apply(
/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/1580462645.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  buyer_late = df.groupby('buyer_tax_id').apply(


In [11]:
cnae_means = df.groupby('main_cnae')['days_to_payment'].mean().reset_index(name='cnae_avg_days')




# CNAE risk tier (based on EDA findings)
def cnae_risk_tier(avg_days):
    if avg_days <= 15: return 'low_risk'
    elif avg_days <= 40: return 'medium_risk'
    elif avg_days <= 70: return 'high_risk'
    else: return 'very_high_risk'

cnae_means['cnae_risk_tier'] = cnae_means['cnae_avg_days'].apply(cnae_risk_tier)

# Transaction count per CNAE (credibility of the average)
cnae_counts = df.groupby('main_cnae').size().reset_index(name='cnae_transaction_count')







In [12]:
# From created_at / due_date
df['created_month'] = df['created_at'].dt.month
df['created_quarter'] = df['created_at'].dt.quarter
df['created_day_of_week'] = df['created_at'].dt.dayofweek
df['created_day_of_month'] = df['created_at'].dt.day
df['created_year'] = df['created_at'].dt.year

# Is end of month? (payments often delayed around month-end)
df['is_month_end'] = df['due_date'].dt.day >= 25

# Days until due date (from creation)
df['days_until_due'] = (df['due_date'] - df['created_at']).dt.days

# Is holiday month? (December/January may have delays)
df['is_holiday_period'] = df['created_month'].isin([12, 1])

In [18]:

# Company age (days since creation)
df['company_age_days'] = (df['created_at'] - pd.to_datetime(df['company_creation_date'])).dt.days


# Is active company?
df['is_active_company'] = np.where(df['company_status'] == 'ATIVA', 1, 0)

# State-level risk (some states may have slower payment patterns)
state_means = df.groupby('state')['days_to_payment'].mean().reset_index(name='state_avg_days')

In [16]:
# Has quod score? (22.8% missing — this itself is predictive)
df['has_quod_score'] = df['quod_score'].notna().astype(int)


In [19]:
# State average days to payment
state_means = df.groupby('state')['days_to_payment'].mean().reset_index(name='state_avg_days')

# State median days (more robust to outliers)
state_median = df.groupby('state')['days_to_payment'].median().reset_index(name='state_median_days')

# State standard deviation (payment volatility by region)
state_std = df.groupby('state')['days_to_payment'].std().reset_index(name='state_std_days')

# State transaction count (credibility of averages)
state_count = df.groupby('state').size().reset_index(name='state_transaction_count')

# State late payment rate (% > 60 days)
state_late_rate = df.groupby('state').apply(
    lambda x: (x['days_to_payment'] > 60).mean()
).reset_index(name='state_late_rate')

# State on-time rate (% <= 30 days)
state_ontime_rate = df.groupby('state').apply(
    lambda x: (x['days_to_payment'] <= 30).mean()
).reset_index(name='state_ontime_rate')

# State average face value (economic indicator)
state_avg_value = df.groupby('state')['face_value'].mean().reset_index(name='state_avg_face_value')

# Merge all state features at once
state_features = state_means.merge(state_median, on='state') \
                            .merge(state_std, on='state') \
                            .merge(state_count, on='state') \
                            .merge(state_late_rate, on='state') \
                            .merge(state_ontime_rate, on='state') \
                            .merge(state_avg_value, on='state')

# Join back to main dataframe
df = df.merge(state_features, on='state', how='left')

/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/1319266190.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_late_rate = df.groupby('state').apply(
/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/1319266190.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_ontime_rate = df.groupby('state').apply(


In [21]:
# Legal nature aggregated features
legal_nature_features = df.groupby('legal_nature').agg(
    legal_nature_avg_days=('days_to_payment', 'mean'),
    legal_nature_median_days=('days_to_payment', 'median'),
    legal_nature_std_days=('days_to_payment', 'std'),
    legal_nature_transaction_count=('days_to_payment', 'count'),
    legal_nature_avg_face_value=('face_value', 'mean'),
    legal_nature_avg_quod=('quod_score', 'mean')
).reset_index()

# Add rate-based features
legal_nature_features['legal_nature_late_rate'] = df.groupby('legal_nature').apply(
    lambda x: (x['days_to_payment'] > 60).mean()
).values

legal_nature_features['legal_nature_ontime_rate'] = df.groupby('legal_nature').apply(
    lambda x: (x['days_to_payment'] <= 30).mean()
).values

# Join back to main dataframe
df = df.merge(legal_nature_features, on='legal_nature', how='left')

/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/3461636072.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  legal_nature_features['legal_nature_late_rate'] = df.groupby('legal_nature').apply(
/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/3461636072.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  legal_nature_features['legal_nature_ontime_rate'] = df.groupby('legal_nature')

In [22]:
df.columns

Index(['asset_id', 'invoice_id', 'face_value', 'buyer_tax_id', 'seller_tax_id',
       'maturity_date', 'due_date', 'settled_at', 'created_at',
       'reference_date', 'days_to_payment', 'payment_status', 'face_value_bin',
       'year', 'month', 'year_month', 'tax_id', 'company_status',
       'company_status_date', 'company_creation_date', 'company_size',
       'main_cnae', 'main_cnae_description', 'secondary_cnae_array',
       'legal_nature', 'is_mei', 'city', 'state', 'zipcode', 'tax_id_quod',
       'quod_score', 'presumed_revenue', 'created_at_quod',
       'max_expected_payment_time', 'buyer_is_first_transaction',
       'created_month', 'created_quarter', 'created_day_of_week',
       'created_day_of_month', 'created_year', 'is_month_end',
       'days_until_due', 'is_holiday_period', 'face_value_log',
       'is_large_transaction', 'has_quod_score', 'company_age_days',
       'is_active_company', 'state_avg_days', 'state_median_days',
       'state_std_days', 'state_transac

In [23]:
# =============================================================================
# LEAKAGE-FREE BUYER FEATURES
# Using only PAST transactions (before current transaction's created_at)
# =============================================================================

# Sort by buyer and date - CRITICAL for point-in-time calculations
df = df.sort_values(['buyer_tax_id', 'created_at']).reset_index(drop=True)

# -----------------------------------------------------------------------------
# 1. HISTORICAL PAYMENT BEHAVIOR (Expanding window, shifted to exclude current)
# -----------------------------------------------------------------------------

# These use .shift(1) to exclude the current row
df['buyer_historical_avg_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().mean().shift(1)
)

df['buyer_historical_median_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().median().shift(1)
)

df['buyer_historical_std_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().std().shift(1)
)

df['buyer_historical_min_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().min().shift(1)
)

df['buyer_historical_max_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().max().shift(1)
)

# Transaction count BEFORE this one
df['buyer_prior_transaction_count'] = df.groupby('buyer_tax_id').cumcount()

# -----------------------------------------------------------------------------
# 2. HISTORICAL FACE VALUE PATTERNS (Safe - uses face_value, not target)
# -----------------------------------------------------------------------------

df['buyer_historical_total_value'] = df.groupby('buyer_tax_id')['face_value'].transform(
    lambda x: x.expanding().sum().shift(1)
)

df['buyer_historical_avg_value'] = df.groupby('buyer_tax_id')['face_value'].transform(
    lambda x: x.expanding().mean().shift(1)
)

df['buyer_historical_max_value'] = df.groupby('buyer_tax_id')['face_value'].transform(
    lambda x: x.expanding().max().shift(1)
)

# -----------------------------------------------------------------------------
# 3. HISTORICAL PAYMENT RELIABILITY RATES
# -----------------------------------------------------------------------------

# On-time rate (<=30 days) from past transactions only
df['_temp_ontime'] = (df['days_to_payment'] <= 30).astype(int)
df['buyer_historical_ontime_rate'] = df.groupby('buyer_tax_id')['_temp_ontime'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Late rate (>60 days) from past transactions only
df['_temp_late'] = (df['days_to_payment'] > 60).astype(int)
df['buyer_historical_late_rate'] = df.groupby('buyer_tax_id')['_temp_late'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Very late rate (>90 days)
df['_temp_very_late'] = (df['days_to_payment'] > 90).astype(int)
df['buyer_historical_very_late_rate'] = df.groupby('buyer_tax_id')['_temp_very_late'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Clean up temp columns
df.drop(columns=['_temp_ontime', '_temp_late', '_temp_very_late'], inplace=True)

# -----------------------------------------------------------------------------
# 4. LAST KNOWN PAYMENT (shifted - uses previous transaction, not current)
# -----------------------------------------------------------------------------

df['buyer_last_payment_days'] = df.groupby('buyer_tax_id')['days_to_payment'].shift(1)
df['buyer_second_last_payment_days'] = df.groupby('buyer_tax_id')['days_to_payment'].shift(2)
df['buyer_third_last_payment_days'] = df.groupby('buyer_tax_id')['days_to_payment'].shift(3)

# Rolling average of last 3 payments (excluding current)
df['buyer_last3_avg_days'] = df.groupby('buyer_tax_id')['days_to_payment'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

# -----------------------------------------------------------------------------
# 5. BUYER FLAGS (Available at prediction time)
# -----------------------------------------------------------------------------

# Is this the buyer's first transaction? (No history available)
df['buyer_is_first_transaction'] = (df['buyer_prior_transaction_count'] == 0).astype(int)

# Is this a repeat buyer?
df['buyer_is_repeat'] = (df['buyer_prior_transaction_count'] > 0).astype(int)

# Is experienced buyer (>5 prior transactions)?
df['buyer_is_experienced'] = (df['buyer_prior_transaction_count'] > 5).astype(int)

# -----------------------------------------------------------------------------
# 6. TENURE FEATURES (Safe - based on dates, not target)
# -----------------------------------------------------------------------------

# Days since buyer's first transaction
df['buyer_first_tx_date'] = df.groupby('buyer_tax_id')['created_at'].transform('first')
df['buyer_days_since_first_tx'] = (df['created_at'] - df['buyer_first_tx_date']).dt.days

# Days since last transaction
df['buyer_last_tx_date'] = df.groupby('buyer_tax_id')['created_at'].shift(1)
df['buyer_days_since_last_tx'] = (df['created_at'] - df['buyer_last_tx_date']).dt.days

# Clean up
df.drop(columns=['buyer_first_tx_date', 'buyer_last_tx_date'], inplace=True)

# -----------------------------------------------------------------------------
# 7. CURRENT TRANSACTION FEATURES (Safe - available at creation time)
# -----------------------------------------------------------------------------

# Face value relative to buyer's historical average
df['face_value_vs_buyer_avg'] = df['face_value'] / (df['buyer_historical_avg_value'] + 1)

# Is this a larger than usual transaction for this buyer?
df['is_above_buyer_avg_value'] = (df['face_value'] > df['buyer_historical_avg_value']).astype(int)

# -----------------------------------------------------------------------------
# 8. TREND FEATURES (Using only historical data)
# -----------------------------------------------------------------------------

def calculate_historical_trend(group):
    """Calculate trend using only past transactions"""
    result = pd.Series(index=group.index, dtype=float)
    result[:] = 0
    
    for i in range(2, len(group)):
        past_payments = group['days_to_payment'].iloc[:i].values
        if len(past_payments) >= 2:
            x = np.arange(len(past_payments))
            slope = np.polyfit(x, past_payments, 1)[0]
            result.iloc[i] = slope
    
    return result

df['buyer_historical_trend'] = df.groupby('buyer_tax_id', group_keys=False).apply(
    calculate_historical_trend
)

print("✅ All buyer features are now leakage-free")

✅ All buyer features are now leakage-free


/var/folders/yj/j2rj2j5s26ndvfgh5k24zbmc0000gn/T/ipykernel_21366/3969138935.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['buyer_historical_trend'] = df.groupby('buyer_tax_id', group_keys=False).apply(


In [25]:
# =============================================================================
# SELLER FEATURES (Leakage-Free)
# Using only PAST transactions for each seller
# =============================================================================

# Ensure sorted by seller and date
df = df.sort_values(['seller_tax_id', 'created_at']).reset_index(drop=True)

# -----------------------------------------------------------------------------
# 1. SELLER HISTORICAL PAYMENT BEHAVIOR (How fast do this seller's customers pay?)
# -----------------------------------------------------------------------------

df['seller_historical_avg_days'] = df.groupby('seller_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().mean().shift(1)
)

df['seller_historical_median_days'] = df.groupby('seller_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().median().shift(1)
)

df['seller_historical_std_days'] = df.groupby('seller_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().std().shift(1)
)

df['seller_historical_min_days'] = df.groupby('seller_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().min().shift(1)
)

df['seller_historical_max_days'] = df.groupby('seller_tax_id')['days_to_payment'].transform(
    lambda x: x.expanding().max().shift(1)
)

# Transaction count before this one
df['seller_prior_transaction_count'] = df.groupby('seller_tax_id').cumcount()

# -----------------------------------------------------------------------------
# 2. SELLER PAYMENT RELIABILITY RATES
# -----------------------------------------------------------------------------

# On-time rate for this seller's customers
df['_temp_ontime'] = (df['days_to_payment'] <= 30).astype(int)
df['seller_historical_ontime_rate'] = df.groupby('seller_tax_id')['_temp_ontime'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Late rate for this seller's customers
df['_temp_late'] = (df['days_to_payment'] > 60).astype(int)
df['seller_historical_late_rate'] = df.groupby('seller_tax_id')['_temp_late'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Very late rate
df['_temp_very_late'] = (df['days_to_payment'] > 90).astype(int)
df['seller_historical_very_late_rate'] = df.groupby('seller_tax_id')['_temp_very_late'].transform(
    lambda x: x.expanding().mean().shift(1)
)

df.drop(columns=['_temp_ontime', '_temp_late', '_temp_very_late'], inplace=True)

# -----------------------------------------------------------------------------
# 3. SELLER TRANSACTION VALUE PATTERNS
# -----------------------------------------------------------------------------

df['seller_historical_total_value'] = df.groupby('seller_tax_id')['face_value'].transform(
    lambda x: x.expanding().sum().shift(1)
)

df['seller_historical_avg_value'] = df.groupby('seller_tax_id')['face_value'].transform(
    lambda x: x.expanding().mean().shift(1)
)

df['seller_historical_max_value'] = df.groupby('seller_tax_id')['face_value'].transform(
    lambda x: x.expanding().max().shift(1)
)

# -----------------------------------------------------------------------------
# 4. SELLER CUSTOMER BASE FEATURES
# -----------------------------------------------------------------------------

# Re-sort by seller for accurate calculations
df = df.sort_values(['seller_tax_id', 'created_at']).reset_index(drop=True)

# Count of unique buyers this seller has worked with (expanding, shifted)
df['seller_unique_buyers'] = df.groupby('seller_tax_id')['buyer_tax_id'].transform(
    lambda x: x.expanding().apply(lambda y: y.nunique(), raw=False).shift(1)
)

# -----------------------------------------------------------------------------
# 5. SELLER FLAGS
# -----------------------------------------------------------------------------

# Is this seller's first transaction?
df['seller_is_first_transaction'] = (df['seller_prior_transaction_count'] == 0).astype(int)

# Is experienced seller (>10 prior transactions)?
df['seller_is_experienced'] = (df['seller_prior_transaction_count'] > 10).astype(int)

# Is high-volume seller (>50 prior transactions)?
df['seller_is_high_volume'] = (df['seller_prior_transaction_count'] > 50).astype(int)

# -----------------------------------------------------------------------------
# 6. SELLER TENURE FEATURES
# -----------------------------------------------------------------------------

# Days since seller's first transaction
df['seller_first_tx_date'] = df.groupby('seller_tax_id')['created_at'].transform('first')
df['seller_days_since_first_tx'] = (df['created_at'] - df['seller_first_tx_date']).dt.days

# Days since seller's last transaction
df['seller_last_tx_date'] = df.groupby('seller_tax_id')['created_at'].shift(1)
df['seller_days_since_last_tx'] = (df['created_at'] - df['seller_last_tx_date']).dt.days

df.drop(columns=['seller_first_tx_date', 'seller_last_tx_date'], inplace=True)

# -----------------------------------------------------------------------------
# 7. SELLER RECENT PERFORMANCE
# -----------------------------------------------------------------------------

# Last known payment time for this seller
df['seller_last_payment_days'] = df.groupby('seller_tax_id')['days_to_payment'].shift(1)

# Rolling average of last 5 payments for this seller
df['seller_last5_avg_days'] = df.groupby('seller_tax_id')['days_to_payment'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# -----------------------------------------------------------------------------
# 8. CURRENT TRANSACTION VS SELLER HISTORY
# -----------------------------------------------------------------------------

# Is this transaction larger than seller's average?
df['face_value_vs_seller_avg'] = df['face_value'] / (df['seller_historical_avg_value'] + 1)

df['is_above_seller_avg_value'] = (df['face_value'] > df['seller_historical_avg_value']).astype(int)

# -----------------------------------------------------------------------------
# 9. BUYER-SELLER RELATIONSHIP FEATURES
# -----------------------------------------------------------------------------

# Re-sort by buyer-seller pair and date
df = df.sort_values(['buyer_tax_id', 'seller_tax_id', 'created_at']).reset_index(drop=True)

# How many times has THIS buyer bought from THIS seller before?
df['buyer_seller_prior_count'] = df.groupby(['buyer_tax_id', 'seller_tax_id']).cumcount()

# Historical avg days for this specific buyer-seller pair
df['buyer_seller_historical_avg_days'] = df.groupby(['buyer_tax_id', 'seller_tax_id'])['days_to_payment'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Is this the first transaction between this buyer and seller?
df['is_first_buyer_seller_tx'] = (df['buyer_seller_prior_count'] == 0).astype(int)

# Is this a repeat buyer-seller relationship?
df['is_repeat_buyer_seller'] = (df['buyer_seller_prior_count'] > 0).astype(int)

# Fill NaN for first transactions
df['seller_historical_std_days'] = df['seller_historical_std_days'].fillna(0)

print("✅ Created seller features:")
seller_cols = [col for col in df.columns if 'seller_' in col or 'buyer_seller' in col]
print(f"   {len(seller_cols)} seller-related features")
for col in seller_cols:
    print(f"   - {col}")

✅ Created seller features:
   27 seller-related features
   - seller_tax_id
   - seller_historical_avg_days
   - seller_historical_median_days
   - seller_historical_std_days
   - seller_historical_min_days
   - seller_historical_max_days
   - seller_prior_transaction_count
   - seller_historical_ontime_rate
   - seller_historical_late_rate
   - seller_historical_very_late_rate
   - seller_historical_total_value
   - seller_historical_avg_value
   - seller_historical_max_value
   - seller_unique_buyers
   - seller_is_first_transaction
   - seller_is_experienced
   - seller_is_high_volume
   - seller_days_since_first_tx
   - seller_days_since_last_tx
   - seller_last_payment_days
   - seller_last5_avg_days
   - face_value_vs_seller_avg
   - is_above_seller_avg_value
   - buyer_seller_prior_count
   - buyer_seller_historical_avg_days
   - is_first_buyer_seller_tx
   - is_repeat_buyer_seller


In [27]:
df.columns

Index(['asset_id', 'invoice_id', 'face_value', 'buyer_tax_id', 'seller_tax_id',
       'maturity_date', 'due_date', 'settled_at', 'created_at',
       'reference_date',
       ...
       'seller_days_since_first_tx', 'seller_days_since_last_tx',
       'seller_last_payment_days', 'seller_last5_avg_days',
       'face_value_vs_seller_avg', 'is_above_seller_avg_value',
       'buyer_seller_prior_count', 'buyer_seller_historical_avg_days',
       'is_first_buyer_seller_tx', 'is_repeat_buyer_seller'],
      dtype='object', length=120)